# SIR Epidemic Model Trainer

This notebook teaches the **SIR compartmental model** — a foundational model in epidemiology that divides a population into Susceptible, Infected, and Recovered compartments.

## Learning Objectives
- Derive the SIR differential equations
- Understand the basic reproduction number $R_0$
- Implement and solve the ODE system numerically
- Compare Euler vs RK4 accuracy for this system
- Extend to the SEIR model

## 1. The SIR Model

The population is divided into three compartments:

| Compartment | Meaning |
|---|---|
| $S$ | Susceptible — can catch the disease |
| $I$ | Infected — currently infectious |
| $R$ | Recovered — immune (or removed) |

The flow is: $S \xrightarrow{\beta S I} I \xrightarrow{\gamma I} R$

The coupled ODEs are:

$$\frac{dS}{dt} = -\beta S I$$

$$\frac{dI}{dt} = \beta S I - \gamma I$$

$$\frac{dR}{dt} = \gamma I$$

where:
- $\beta$ = transmission rate (how fast susceptibles become infected)
- $\gamma$ = recovery rate ($1/\gamma$ = average infection duration)

**Key invariant:** $\frac{d}{dt}(S + I + R) = 0$, so total population $N = S + I + R$ is conserved.

## 2. The Basic Reproduction Number $R_0$

The **basic reproduction number** is:

$$R_0 = \frac{\beta N}{\gamma}$$

This represents the average number of secondary infections caused by one infected individual in a fully susceptible population.

- If $R_0 > 1$: the epidemic grows (each infected person infects more than one other)
- If $R_0 < 1$: the epidemic dies out
- If $R_0 = 1$: endemic equilibrium

The epidemic peaks when $\frac{dI}{dt} = 0$, which occurs when $S = \gamma / \beta = N / R_0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Population parameters
N = 50000
S0 = 45400
I0 = 2100
R0_init = 2500  # initial recovered (not R0 the reproduction number!)

# Epidemic parameters
beta = 0.00001   # Transmission rate (pre-normalized)
gamma = 1.0 / 14  # Recovery rate (14-day illness)

# Compute R0
R0 = beta * N / gamma
print(f"Basic reproduction number R₀ = {R0:.2f}")
print(f"Average infection duration: {1/gamma:.0f} days")
print(f"Epidemic will {'grow' if R0 > 1 else 'die out'}")
print(f"Peak occurs when S = {N/R0:.0f}")

## 3. Building the ODE System

Let's implement the SIR derivatives from scratch:

In [ ]:
def sir_derivatives(state, t):
    """Compute dS/dt, dI/dt, dR/dt for the SIR model."""
    S, I, R = state
    
    dS = -beta * S * I
    dI = beta * S * I - gamma * I
    dR = gamma * I
    
    return np.array([dS, dI, dR])

# Verify conservation
state0 = np.array([S0, I0, R0_init], dtype=np.float64)
derivs = sir_derivatives(state0, 0.0)
print(f"Sum of derivatives: {np.sum(derivs):.2e} (should be ~0)")
print(f"dS/dt = {derivs[0]:.1f}, dI/dt = {derivs[1]:.1f}, dR/dt = {derivs[2]:.1f}")

In [ ]:
def euler_step(state, t, dt, derivs_fn):
    """Single Euler step."""
    return state + dt * derivs_fn(state, t)

def rk4_step(state, t, dt, derivs_fn):
    """Single RK4 step."""
    k1 = derivs_fn(state, t)
    k2 = derivs_fn(state + 0.5*dt*k1, t + 0.5*dt)
    k3 = derivs_fn(state + 0.5*dt*k2, t + 0.5*dt)
    k4 = derivs_fn(state + dt*k3, t + dt)
    return state + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)

def simulate_sir(state0, dt, n_days, step_fn):
    """Simulate SIR for n_days."""
    n_steps = int(n_days / dt)
    history = np.zeros((n_steps + 1, 3))
    history[0] = state0
    t = 0.0
    for i in range(n_steps):
        history[i + 1] = step_fn(history[i], t, dt, sir_derivatives)
        t += dt
    times = np.linspace(0, n_days, n_steps + 1)
    return times, history

In [ ]:
# Simulate for 200 days
dt = 0.1  # days
n_days = 200

times, sir_history = simulate_sir(state0, dt, n_days, rk4_step)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(times, sir_history[:, 0], 'b-', label='Susceptible', linewidth=2)
ax.plot(times, sir_history[:, 1], 'r-', label='Infected', linewidth=2)
ax.plot(times, sir_history[:, 2], 'g-', label='Recovered', linewidth=2)
ax.axhline(y=N/R0, color='gray', linestyle='--', alpha=0.5,
           label=f'S = N/R₀ = {N/R0:.0f} (peak threshold)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Population')
ax.set_title(f'SIR Model (R₀ = {R0:.1f}, N = {N})')
ax.legend(loc='right')
ax.grid(True, alpha=0.3)
plt.show()

# Find peak infection
peak_idx = np.argmax(sir_history[:, 1])
print(f"Peak infection: {sir_history[peak_idx, 1]:.0f} at day {times[peak_idx]:.1f}")
print(f"Total ever infected: {N - sir_history[-1, 0]:.0f}")

## 4. Euler vs RK4 Accuracy Comparison

Let's compare how well each method conserves the population invariant $N = S + I + R$:

In [ ]:
dt_values = [1.0, 0.5, 0.1, 0.01]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for dt_val in dt_values:
    t_e, h_e = simulate_sir(state0, dt_val, 200, euler_step)
    t_r, h_r = simulate_sir(state0, dt_val, 200, rk4_step)
    
    # Population conservation error
    pop_error_euler = np.abs(np.sum(h_e, axis=1) - N)
    pop_error_rk4 = np.abs(np.sum(h_r, axis=1) - N)
    
    axes[0].plot(t_e, pop_error_euler, label=f'Euler dt={dt_val}')
    axes[1].plot(t_r, pop_error_rk4, label=f'RK4 dt={dt_val}')

axes[0].set_xlabel('Time (days)')
axes[0].set_ylabel('|S+I+R - N|')
axes[0].set_title('Euler: Population Conservation Error')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (days)')
axes[1].set_ylabel('|S+I+R - N|')
axes[1].set_title('RK4: Population Conservation Error')
axes[1].legend()
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Using the `physics_modeling` Package

In [ ]:
from physics_modeling.epidemics.sir import SIRConfig, SIRSimulation

config = SIRConfig(
    S0=45400, I0=2100, R0=2500,
    beta=0.00001, gamma=1/14,
    dt=0.1, integrator="rk4",
)

sim = SIRSimulation(config)
print(f"Total population: {sim.total_population}")
print(f"R₀ = {sim.basic_reproduction_number:.2f}")
print(f"Initial state: S={sim.state[0]:.0f}, I={sim.state[1]:.0f}, R={sim.state[2]:.0f}")

# Run for 200 days
n_steps = 2000
history = np.zeros((n_steps + 1, 3))
history[0] = sim.state.copy()

for i in range(n_steps):
    history[i + 1] = sim.step()

print(f"\nAfter 200 days:")
print(f"  S = {sim.state[0]:.0f}")
print(f"  I = {sim.state[1]:.0f}")
print(f"  R = {sim.state[2]:.0f}")
print(f"  Total = {sim.total_population:.2f} (should be {N})")

## 6. Effect of $R_0$ on Epidemic Dynamics

Let's vary the transmission rate to see how $R_0$ affects the epidemic curve:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

beta_values = [0.000005, 0.00001, 0.00002, 0.00004]
colors = ['green', 'blue', 'orange', 'red']

for b_val, color in zip(beta_values, colors):
    r0_val = b_val * N / gamma
    cfg = SIRConfig(S0=49000, I0=1000, R0=0, beta=b_val, gamma=gamma, dt=0.1)
    s = SIRSimulation(cfg)
    
    infected = np.zeros(2000)
    for i in range(2000):
        infected[i] = s.step()[1]
    
    t = np.arange(2000) * 0.1
    ax.plot(t, infected, color=color, linewidth=2,
            label=f'R₀ = {r0_val:.1f}')

ax.set_xlabel('Time (days)')
ax.set_ylabel('Infected')
ax.set_title('Effect of R₀ on Epidemic Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Exercise: Implement the SEIR Model

The **SEIR model** adds an **Exposed** compartment — individuals who are infected but not yet infectious:

$$S \xrightarrow{\beta S I} E \xrightarrow{\sigma E} I \xrightarrow{\gamma I} R$$

The equations become:

$$\frac{dS}{dt} = -\beta S I$$
$$\frac{dE}{dt} = \beta S I - \sigma E$$
$$\frac{dI}{dt} = \sigma E - \gamma I$$
$$\frac{dR}{dt} = \gamma I$$

where $\sigma$ is the rate at which exposed individuals become infectious ($1/\sigma$ = incubation period).

**Tasks:**
1. Implement `seir_derivatives(state, t)` with state = `[S, E, I, R]`
2. Use $\sigma = 1/5$ (5-day incubation period)
3. Simulate and plot all four compartments
4. Compare the peak timing with the SIR model — is it delayed?

In [ ]:
# Your SEIR implementation here
sigma = 1.0 / 5.0  # Incubation rate (5-day incubation period)

def seir_derivatives(state, t):
    """Compute derivatives for the SEIR model."""
    S, E, I, R = state
    
    dS = -beta * S * I
    dE = beta * S * I - sigma * E
    dI = sigma * E - gamma * I
    dR = gamma * I
    
    return np.array([dS, dE, dI, dR])

# TODO: Set initial state [S0, E0, I0, R0] and simulate
# Hint: Start with E0=500, I0=100, and compare with SIR peak timing